# NF-UQ-NIDS-v3 — 1-Stage Gaussian Fuzzy Residual Gate-BiLSTM + Global KMeans + SMOTE

**Converted from:** `bilstm-2stage-fuzzy-residualgate`

### Thay đổi chính:
| Hạng mục | 2-Stage | 1-Stage |
|----------|---------|-----------|
| Architecture | S1 Binary + S2 Multiclass | ✅ Single Multiclass model |
| GaussianFuzzyResidualGate | Giữ nguyên | ✅ Giữ nguyên (copy từ S2) |
| BiLSTM input | 256 hidden (S2) | ✅ 256 hidden |
| Output classes | S1: 2 + S2: K attack | ✅ 1+K (tất cả trong 1 model) |
| Training data | Cap 200K (S1) + 150K (S2) | ✅ Cap 150K + SMOTE |
| Validation | Binary + Multiclass | ✅ Multiclass only |
| Test inference | 2-stage pipeline (S1→S2) | ✅ Single pass |
| Timing | S1 + S2 (conditional) | ✅ Single model latency |

### Kiến trúc 1-Stage Gaussian Fuzzy Residual Gate-BiLSTM:
```
Input(B,F)
  ↓ GaussianFuzzyResidualGate → y(B,F)   [weight = exp(-dist/2σ²) / max]
  ↓ Reshape → (B, seq, input_sz)
  ↓ InputProjection → (B, seq, H)
  ↓ Multi-head Self-Attention → (B, seq, H)   ← trước LSTM
  ↓ BiLSTM → (B, seq, H×2)
  ↓ FuzzyAttentionPooling → (B, H×2)
  ↓ Attention-weighted fuzzy residual → (B, H×2 + F)
  ↓ Head → Benign + Attack classes
```

**GaussianFuzzyResidualGate (copy từ mlp-gaussian-fuzzy-attention-v5):**
- `center = zeros(n_features)` — learnable
- `log_sigma` init sao cho `softplus(log_sigma) ≈ 1` — learnable
- Forward: `sigma = softplus(log_sigma)+1e-6`, `weight = exp(-(x-center)²/2σ²)`, normalize by max, `x_attn = x * weight`
- Output: `(B, n_features)` — cùng chiều input, không tăng chiều
- BiLSTM n_in = `n_features` (không cần concat, không embed)

In [ ]:
!pip install scikit-learn imbalanced-learn pyarrow pandas numpy torch matplotlib seaborn tqdm -q
print('✅ Thư viện đã cài xong')

In [ ]:
import os, warnings, gc, time, pickle, json, copy, math
import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                              confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import label_binarize, LabelEncoder
from sklearn.cluster import MiniBatchKMeans
from imblearn.over_sampling import SMOTE
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

DATASET_DIR = '/kaggle/input/datasets/nguyennhuwz/nf-uq-nids-v3-shuffle-preprocessed'

TRAIN_PATH         = os.path.join(DATASET_DIR, 'train_processed.parquet')
VAL_PATH           = os.path.join(DATASET_DIR, 'val_processed.parquet')
TEST_PATH          = os.path.join(DATASET_DIR, 'test_processed.parquet')
CLASS_MAPPING_PATH = os.path.join(DATASET_DIR, 'class_mapping.pkl')
CLASS_WEIGHTS_PATH = os.path.join(DATASET_DIR, 'class_weights.json')
CONSTANT_COLS_PATH = os.path.join(DATASET_DIR, 'constant_cols.pkl')
SCALER_PATH        = os.path.join(DATASET_DIR, 'quantile_scaler.pkl')

OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

def ram_gb():      return psutil.virtual_memory().used / 1e9
def ram_free_gb(): return psutil.virtual_memory().available / 1e9

if os.path.exists(SCALER_PATH):
    with open(SCALER_PATH, 'rb') as f:
        SCALER = pickle.load(f)
    print(f'✅ Loaded quantile_scaler: {type(SCALER).__name__}')
else:
    SCALER = None
    print('⚠️  quantile_scaler.pkl không tìm thấy!')

with open(CLASS_MAPPING_PATH, 'rb') as f:
    class_mapping = pickle.load(f)
with open(CLASS_WEIGHTS_PATH, 'r') as f:
    class_weights_raw = json.load(f)

constant_cols = []
if os.path.exists(CONSTANT_COLS_PATH):
    with open(CONSTANT_COLS_PATH, 'rb') as f:
        constant_cols = pickle.load(f)
    print(f'  constant_cols ({len(constant_cols)}): {constant_cols[:10]}')

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, CLASS_MAPPING_PATH, CLASS_WEIGHTS_PATH, SCALER_PATH]:
    size   = os.path.getsize(p) / 1024**2 if os.path.exists(p) else -1
    status = f'✅ {size:.1f} MB' if os.path.exists(p) else '❌ NOT FOUND'
    print(f'  {status}  {os.path.basename(p)}')